In [ ]:
import os
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

DATA_ROOT  = "<DATASET_ROOT>"
# Replace this placeholder with your local dataset root directory.
CSV_PATH   = os.path.join(DATA_ROOT, "<DATASET_ANNOTATION_CSV>")
# Replace this placeholder with your dataset annotation CSV file name.
SAVE_DIR   = "<MODEL_ROOT>"
DATA_TYPE  = 'disp'      # 'disp' or 'acc'
BATCH_SIZE = 32
LR         = 1e-4
EPOCHS     = 40
NUM_WORKERS = 4
SEED       = 42
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, "checkpoints"), exist_ok=True)
print(f"Device: {DEVICE} | DataType: {DATA_TYPE}")


In [ ]:
class DoubleConv1D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 3, padding=1),
            # nn.BatchNorm1d(out_ch),
            # nn.ReLU(inplace=True),
            nn.Conv1d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


class DeterministicUNet(nn.Module):
    """
    Deterministic U-Net baseline.

    Input: floor response condition [B, C_y, T].
    Output: predicted ground motion [B, 2, T].
    No diffusion process is used; this is direct end-to-end regression.
    """
    def __init__(self, condition_channels=6, ground_channels=2):
        super().__init__()
        self.inc   = DoubleConv1D(condition_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool1d(2), DoubleConv1D(64,  128))
        self.down2 = nn.Sequential(nn.MaxPool1d(2), DoubleConv1D(128, 256))
        self.bot   = nn.Sequential(nn.MaxPool1d(2), DoubleConv1D(256, 512))

        self.up1   = nn.ConvTranspose1d(512, 256, 2, stride=2)
        self.dec1  = DoubleConv1D(512, 256)
        self.up2   = nn.ConvTranspose1d(256, 128, 2, stride=2)
        self.dec2  = DoubleConv1D(256, 128)
        self.up3   = nn.ConvTranspose1d(128, 64,  2, stride=2)
        self.dec3  = DoubleConv1D(128, 64)
        self.outc  = nn.Conv1d(64, ground_channels, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.bot(x3)

        def _up(feat, skip, up, dec):
            feat = up(feat)
            if feat.size(2) != skip.size(2):
                feat = F.interpolate(feat, size=skip.size(2))
            return dec(torch.cat([skip, feat], dim=1))

        x = _up(x4, x3, self.up1, self.dec1)
        x = _up(x,  x2, self.up2, self.dec2)
        x = _up(x,  x1, self.up3, self.dec3)
        return self.outc(x)

print("Model defined.")

In [ ]:
import sys
sys.path.append("<PROJECT_ROOT>")
# Replace this placeholder with your local project root if needed.
from dataset import SeismicDataset

train_ds = SeismicDataset(
    DATA_ROOT, CSV_PATH,
    data_type=DATA_TYPE, layer_combo=[0,1,2],
    split='train',
)
val_ds = SeismicDataset(
    DATA_ROOT, CSV_PATH,
    data_type=DATA_TYPE, layer_combo=[0,1,2],
    split='val', external_scale=train_ds.scale,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)

print(f"Train: {len(train_ds)} samples | Val: {len(val_ds)} samples")
print(f"Scale: {train_ds.scale:.4f}")

In [ ]:
torch.manual_seed(SEED)

model     = DeterministicUNet(
    condition_channels=train_ds.condition_channels,
    ground_channels=2
).to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=LR)
criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 'min', patience=10, factor=0.5
)

best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    # Training
    model.train()
    total = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        layers = batch['layers'].to(DEVICE)
        ground = batch['ground'].to(DEVICE)
        pred   = model(layers)
        loss   = criterion(pred, ground)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    train_loss = total / len(train_loader)
    train_losses.append(train_loss)

    # Validation
    model.eval()
    val_total = 0.0
    with torch.no_grad():
        for batch in val_loader:
            layers = batch['layers'].to(DEVICE)
            ground = batch['ground'].to(DEVICE)
            pred   = model(layers)
            val_total += criterion(pred, ground).item()
    val_loss = val_total / len(val_loader)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(),
                   os.path.join(SAVE_DIR, "checkpoints", "best_model.pth"))

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train={train_loss:.6f} | val={val_loss:.6f} | "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")

print(f"\nBest val loss: {best_val_loss:.6f}")

In [ ]:
from reconstruct_full import reconstruct_all

model.load_state_dict(
    torch.load(os.path.join(SAVE_DIR, "checkpoints", "best_model.pth"),
               map_location=DEVICE, weights_only=True)
)
model.eval()

# Wrap the deterministic model with the DDIMSampler-compatible interface.
class DeterministicSampler:
    """Wrap the deterministic U-Net with the same interface as DDIMSampler."""
    @torch.no_grad()
    def sample(self, model, condition, device, eta=0.0, verbose=False):
        return model(condition)

sampler = DeterministicSampler()

reconstruct_all(
    model=model,
    ddim_sampler=sampler,
    val_ds=val_ds,
    device=DEVICE,
    save_dir=SAVE_DIR,
    fs=100.0,
    data_type=DATA_TYPE,
)